# CSE465 ColorBench — Generator–Verifier–Oracle Adaptive Skill Pipeline

**Architecture:** Generator–Verifier–Oracle (GVO) Architecture & Baseline Ablation Suite
- **Experiment Modes:**
  1. `baseline`: Direct VQA (zero skills, direct forward pass)
  2. `ace_baseline`: Generator -> Solver -> Reflector -> Solver (legacy ACE baseline)
  3. `gen_verifier`: Generator <-> Verifier iterative feedback loop (Ablation Mode C)
  4. `gen_verifier_oracle`: Generator <-> Verifier + Hidden Oracle Generalization Gating (Full Architecture)
- **Information Boundary Enforcement:** Target Ground Truth is strictly isolated for post-hoc evaluation only; Verifier uses private validation instances; Oracle uses disjoint hidden generalization instances.

**Target:** Google Colab Free-Tier (Tesla T4 GPU, 15GB VRAM) — 100% Local Inference (0 API Keys)


## 1. Mount Google Drive


In [ ]:
from google.colab import drive
import os
from datetime import datetime

# Mount Drive
drive.mount('/content/drive')

# Set up Run Tag and Directory
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S") + "_GVO_Pipeline"
DRIVE_SAVE_DIR = f"/content/drive/MyDrive/CSE465_Results/{RUN_TAG}"

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"\nAll results will be saved to Google Drive: {DRIVE_SAVE_DIR}")


## 2. Install Dependencies


In [ ]:
!pip install -q transformers bitsandbytes accelerate qwen-vl-utils datasets "pillow<11.0.0"


## 3. Clone Repository


In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/cse465-project.git"  # <-- UPDATE THIS

# Go back to /content before cloning
%cd /content
if not os.path.exists("cse465-project"):
    !git clone {REPO_URL}

%cd cse465-project


## 4. Verify GPU


In [ ]:
import torch

# GPU Verification
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} ({vram:.1f} GB VRAM) — Ready for 100% Local Execution.")
else:
    print("WARNING: No GPU detected! Set runtime type to GPU (T4).")


## 5. Load Models & Initialize ACE Orchestrator

Loads Qwen2.5-VL-7B in 4-bit precision and initializes Generator, Verifier, Oracle, and Legacy Reflector.


In [ ]:
from qwen_solver import QwenSolver
from ace import ACE

# Load Qwen2.5-VL-7B in 4-bit
solver = QwenSolver()
solver.load_model()

# Initialize Orchestrator
ace_system = ACE(solver=solver)

print("\nAll models and GVO agents ready! Running 100% locally on Colab GPU.")


## 6. Task Selection & Deterministic Dataset Partitioning

Select the task and mode. In G-V and G-V-O modes, dataset is partitioned into:
- **5 Verifier instances** ($P_{verifier}$)
- **5 Oracle instances** ($P_{oracle}$)
- **Evaluation Stream instances**


In [ ]:
#@title ⚙️ Experiment Configuration { run: "auto" }

SELECTED_TASK = "Color Illusion" #@param ["Color Recognition", "Color Illusion", "Color Mimicry", "Color Counting", "Color Comparison", "Color Blindness", "Object Counting", "Color Proportion", "Object Recognition", "Color Extraction"]
EXPERIMENT_MODE = "gen_verifier_oracle" #@param ["baseline", "ace_baseline", "gen_verifier", "gen_verifier_oracle"]
ITERATIONS = 3 #@param {type:"integer"}
ORACLE_RETRIES = 2 #@param {type:"integer"}
QUICK_TEST = False #@param {type:"boolean"}
TEST_LIMIT = 5 #@param {type:"integer"}

from data_loader import ColorBenchDataLoader, partition_task_dataset

TASK_SLUG = SELECTED_TASK.lower().replace(" ", "_")
loader = ColorBenchDataLoader(task_filter=SELECTED_TASK)
all_items = list(loader.stream_instances())

if EXPERIMENT_MODE in ["gen_verifier", "gen_verifier_oracle"]:
    verifier_pool, oracle_suite, eval_instances = partition_task_dataset(all_items)
    print(f"Partitioned: {len(verifier_pool)} Verifier pool, {len(oracle_suite)} Oracle suite, {len(eval_instances)} Eval items.")
else:
    verifier_pool, oracle_suite = [], []
    eval_instances = all_items

limit = min(TEST_LIMIT, len(eval_instances)) if QUICK_TEST else len(eval_instances)

print(f"\n{'='*75}")
print(f"SELECTED TASK: {SELECTED_TASK} | MODE: {EXPERIMENT_MODE}")
print(f"Total Task Instances: {len(all_items)} | Evaluating on: {limit} instances.")
print(f"Inner Iterations: {ITERATIONS} | Oracle Retries: {ORACLE_RETRIES}")
print(f"{'='*75}\n")


## 7. Run Experiment Pipeline


In [ ]:
import os
import time
import json
from data_loader import IncrementalLogger

output_filename = f"results_{TASK_SLUG}_{EXPERIMENT_MODE}.jsonl"
output_path = os.path.join(DRIVE_SAVE_DIR, output_filename)
logger = IncrementalLogger(output_path)

correct, total = 0, 0
start = time.time()

print(f"\n{'='*75}")
print(f"STARTING PIPELINE [{EXPERIMENT_MODE.upper()}] ({SELECTED_TASK} — {limit} instances)")
print(f"Logging to: {output_path}")
print(f"{'='*75}\n")

for item in eval_instances[:limit]:
    if item["idx"] in logger.processed_indices:
        continue

    q_start = time.time()
    question = item["question"]
    choices = item["choices"]
    image = item["image"]
    gt = item["answer"]

    print(f"[{total + 1}/{limit}] Target idx={item['idx']} (ID: {item['id']}): {question}")

    if EXPERIMENT_MODE == "gen_verifier_oracle":
        res = ace_system.run_gvo_pipeline(
            target_item=item,
            verifier_pool=verifier_pool,
            oracle_suite=oracle_suite,
            max_inner_iters=ITERATIONS,
            max_oracle_retries=ORACLE_RETRIES,
        )
    elif EXPERIMENT_MODE == "gen_verifier":
        res = ace_system.run_gv_pipeline(
            target_item=item,
            verifier_pool=verifier_pool,
            max_inner_iters=ITERATIONS,
        )
    elif EXPERIMENT_MODE == "ace_baseline":
        plan = ace_system.generate_skill(question, choices, image=image)
        skill = plan.get("skill", "")
        sol = solver.solve(image, question, choices, mode="adaptive_skills", skill=skill)
        pred, raw = sol["prediction"], sol["raw_output"]
        all_iters = [{"iteration": 1, "skill": skill, "prediction": pred}]
        for i in range(2, ITERATIONS + 1):
            refl = ace_system.reflect_and_refine(question, choices, skill, pred, raw, image=image)
            skill = refl.get("skill", skill)
            sol = solver.solve(image, question, choices, mode="adaptive_skills", skill=skill)
            pred, raw = sol["prediction"], sol["raw_output"]
            all_iters.append({"iteration": i, "skill": skill, "prediction": pred})
        res = {"prediction": pred, "skill": skill, "iterations": all_iters}
    else: # baseline
        sol = solver.solve(image, question, choices, mode="baseline")
        res = {"prediction": sol["prediction"], "raw_output": sol["raw_output"]}

    pred = res["prediction"]
    is_correct = (pred.strip().upper() == gt.strip().upper())
    if is_correct: correct += 1
    total += 1

    q_time = time.time() - q_start
    status = 'CORRECT' if is_correct else f'WRONG (Expected {gt})'
    print(f"  -> Final Prediction: {pred} | GT: {gt} [{status}] (took {q_time:.1f}s)")
    if "oracle_verdict" in res:
        print(f"  -> Oracle: {res['oracle_verdict']} (Acc: {res['oracle_accuracy']:.2f}, Retries: {res.get('oracle_retries', 0)})")
    print(f"  -> Running Score: {correct}/{total} ({correct/total*100:.1f}%)\n")

    logger.log_result({
        "idx": item["idx"], "id": item["id"], "task": item["task"],
        "question": question, "choices": choices,
        "ground_truth": gt, "prediction": pred, "is_correct": is_correct,
        "mode": EXPERIMENT_MODE,
        "skill": res.get("skill"),
        "oracle_verdict": res.get("oracle_verdict"),
        "oracle_accuracy": res.get("oracle_accuracy"),
        "oracle_retries": res.get("oracle_retries"),
        "iterations": res.get("iterations"),
        "oracle_attempts": res.get("oracle_attempts"),
    })

elapsed = time.time() - start
print(f"\n{'='*75}")
print(f"{EXPERIMENT_MODE.upper()} COMPLETED ({SELECTED_TASK}): {correct}/{total} ({correct/total*100:.2f}%) in {elapsed:.1f}s")
print(f"Saved to: {output_path}")
print(f"{'='*75}\n")


## 8. Comparative Analysis & Reporting

Evaluates and prints side-by-side comparative accuracy tables across all completed experiment modes.


In [ ]:
!python eval_results.py --dir "{DRIVE_SAVE_DIR}"
